In [ ]:
import sys
import os
# 将项目根目录加入 path，以便导入 core 等模块
sys.path.append(os.path.abspath(".."))
from core import enable_logging
enable_logging()
import json
from core.history import CanonicalMessage, CanonicalBlock

from core.llm import EasyLLM
from context.token.counter import TokenCounter
from context.compressor.history import (
    RuleBasedHistoryCompactor,
    LLMHistoryCompactor,
    
)


In [2]:
llm = EasyLLM(
    provider="openai",
    base_url="http://127.0.0.1:5124/v1",
    api_key="122",
    model="qwen3.5-9b",
)

2026-04-25 00:20:37,491 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b


In [3]:
token_counter = TokenCounter()


In [4]:
def create_mock_history():
    history = []
    
    # 轮次 1
    history.append(CanonicalMessage(
        role="user", 
        content=[CanonicalBlock(type="text", text="帮我搜索一下本地的文档，找一下关于EasyAgent的架构说明。")]
    ))
    history.append(CanonicalMessage(
        role="assistant",
        content=[
            CanonicalBlock(type="text", text="好的，我先用 grep_search 工具搜索一下相关的文档。"),
            CanonicalBlock(type="function_call", name="grep_search", arguments='{"query": "架构", "path": "./docs"}')
        ]
    ))
    history.append(CanonicalMessage(
        role="tool",
        content=[
            CanonicalBlock(type="function_response", call_id="call_1", name="grep_search", output="匹配到 1000 个结果：\n" + ("非常冗长的架构设计文档内容... \n" * 500))
        ]
    ))
    
    # 轮次 2
    history.append(CanonicalMessage(
        role="user", 
        content=[CanonicalBlock(type="text", text="帮我看看 manager.py 是怎么写的。")]
    ))
    history.append(CanonicalMessage(
        role="assistant",
        content=[
            CanonicalBlock(type="text", text="这就去查看源码。"),
            CanonicalBlock(type="function_call", name="view_file", arguments='{"path": "manager.py"}')
        ]
    ))
    history.append(CanonicalMessage(
        role="tool",
        content=[
            CanonicalBlock(type="function_response", call_id="call_2", name="view_file", output="class ContextManager:\n" + ("    def some_method(self):\n        pass\n" * 400))
        ]
    ))
    
    # 轮次 3 (最近一轮，应该被保护)
    history.append(CanonicalMessage(
        role="user", 
        content=[CanonicalBlock(type="text", text="根据刚才看到的源码，写一个总结。")]
    ))
    
    return history


In [5]:
history = create_mock_history()
print(f"原始历史消息总数: {len(history)}")
print(f"原始历史字数 (近似 Token): {len(json.dumps([res.to_dict() for res in history], ensure_ascii=False))}")


原始历史消息总数: 7
原始历史字数 (近似 Token): 27973


In [6]:
rule_compactor = RuleBasedHistoryCompactor(token_counter=token_counter, recent_turns=1)


In [7]:
print("开始纯规则压缩...")
rule_result = rule_compactor.compact(history, max_tokens=10000)

print(f"\\n规则压缩完成！结果消息数: {len(rule_result)}")
print(f"结果字数 (近似 Token): {len(json.dumps(rule_result, ensure_ascii=False))}")
print("\\n【规则压缩后的前几条消息】 (注意查看它是如何被截断成骨架摘要的)：")
print(json.dumps(rule_result, indent=2, ensure_ascii=False))
print("\\n运行信息:", json.dumps(rule_compactor.get_last_run_info(), indent=2))


开始纯规则压缩...
\n规则压缩完成！结果消息数: 7
结果字数 (近似 Token): 2690
\n【规则压缩后的前几条消息】 (注意查看它是如何被截断成骨架摘要的)：
[
  {
    "record_type": "canonical_message",
    "role": "user",
    "content": [
      {
        "type": "text",
        "text": "帮我搜索一下本地的文档，找一下关于EasyAgent的架构说明。",
        "metadata": {}
      }
    ],
    "time": "2026-04-24 23:51:09.181705",
    "metadata": {}
  },
  {
    "record_type": "canonical_message",
    "role": "assistant",
    "content": [
      {
        "type": "text",
        "text": "好的，我先用 grep_search 工具搜索一下相关的文档。",
        "metadata": {}
      },
      {
        "type": "function_call",
        "name": "grep_search",
        "arguments": "{\"query\": \"架构\", \"path\": \"./docs\"}",
        "metadata": {}
      }
    ],
    "time": "2026-04-24 23:51:09.181718",
    "metadata": {}
  },
  {
    "record_type": "canonical_message",
    "role": "tool",
    "content": [
      {
        "type": "function_response",
        "call_id": "call_1",
        "name": "grep_search",
        "out

In [17]:
# hybrid_compactor = HybridHistoryCompactor(
#     llm=llm, 
#     token_counter=token_counter, 
#     recent_turns=1
# )

from context.compressor.history import LLMHistoryCompactor
lmmcom = LLMHistoryCompactor(
    llm=llm, 
    token_counter=token_counter, 
    recent_turns=1
)
print("===== 场景 A: 预算充足 (10000 tokens) =====")
hybrid_result_A = lmmcom.compact(history, max_tokens=10000)
print(f"\\n混合压缩完成 (A)！结果消息数: {len(hybrid_result_A)}")
print("运行信息:", json.dumps(lmmcom.get_last_run_info(), indent=2))



2026-04-25 00:16:06,716 | INFO | Compact History by LLM


===== 场景 A: 预算充足 (10000 tokens) =====


2026-04-25 00:16:25,441 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


\n混合压缩完成 (A)！结果消息数: 4
运行信息: {
  "compactor": "LLMHistoryCompactor",
  "status": "success",
  "fallback_used": false,
  "mode": "sync",
  "input_messages": 7,
  "output_messages": 4
}


In [18]:
print(json.dumps(hybrid_result_A , indent=2, ensure_ascii=False))


[
  {
    "record_type": "canonical_message",
    "role": "assistant",
    "provider": "history_compactor",
    "provider_message_type": "summary",
    "content": [
      {
        "type": "text",
        "text": "用户请求搜索本地文档中关于 EasyAgent 的架构说明，助手调用 grep_search 工具在./docs 中查询。"
      }
    ],
    "metadata": {
      "compacted_summary": true
    }
  },
  {
    "record_type": "canonical_message",
    "role": "assistant",
    "provider": "history_compactor",
    "provider_message_type": "summary",
    "content": [
      {
        "type": "text",
        "text": "工具返回 grep_search 匹配到 1000 个冗长内容，随后用户要求查看 manager.py 源码。"
      }
    ],
    "metadata": {
      "compacted_summary": true
    }
  },
  {
    "record_type": "canonical_message",
    "role": "assistant",
    "provider": "history_compactor",
    "provider_message_type": "summary",
    "content": [
      {
        "type": "text",
        "text": "助手调用 view_file 工具查看 manager.py，工具返回 ContextManager 类代码片段。"
      }
    ],
    "metadata": 

In [6]:
hybrid_compactor = HybridHistoryCompactor(
    llm=llm, 
    token_counter=token_counter, 
    recent_turns=1
)

print("===== 场景 A: 预算充足 (10000 tokens) =====")
hybrid_result_A = hybrid_compactor.compact(history, max_tokens=10000)
print(f"\\n混合压缩完成 (A)！结果消息数: {len(hybrid_result_A)}")
print("运行信息:", json.dumps(hybrid_compactor.get_last_run_info(), indent=2))

print("\\n===== 场景 B: 预算紧张 (200 tokens) =====")
# 故意把 max_tokens 设得很小，逼迫它使用大模型
hybrid_result_B = hybrid_compactor.compact(history, max_tokens=200)
print(f"\\n混合压缩完成 (B)！结果消息数: {len(hybrid_result_B)}")
# print("运行信息:", json.dumps(hybrid_compactor.get_last_run_info(), indent=2))
# print("\\n【大模型深度脱水后的终极摘要】：")
# print(json.dumps(hybrid_result_B, indent=2, ensure_ascii=False))

2026-04-25 00:20:48,435 | INFO | Compact History by truncating tool messages
2026-04-25 00:20:48,446 | INFO | Hybrid: Compact History by Rule is sufficient.
2026-04-25 00:20:48,448 | INFO | Compact History by truncating tool messages
2026-04-25 00:20:48,449 | INFO | Hybrid: Rule-based is over budget, using LLM Compactor.
2026-04-25 00:20:48,450 | INFO | Compact History by LLM


===== 场景 A: 预算充足 (10000 tokens) =====
\n混合压缩完成 (A)！结果消息数: 7
运行信息: {
  "compactor": "RuleBasedHistoryCompactor",
  "status": "success",
  "fallback_used": false,
  "recent_turns": 1,
  "input_messages": 7,
  "output_messages": 7,
  "strategy": "rule"
}
\n===== 场景 B: 预算紧张 (200 tokens) =====


2026-04-25 00:21:07,893 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


\n混合压缩完成 (B)！结果消息数: 5
